In [ ]:
import requests
import pandas as pd
import numpy as np
from gprofiler import GProfiler
from collections import defaultdict
from typing import List, Dict, Any, Optional
import spacy
import xml.etree.ElementTree as ET
import re
import torch
import time
from tqdm import tqdm
from multiprocessing.dummy import Pool as ThreadPool
from transformers import AutoTokenizer, AutoModelForSequenceClassification

nlp = spacy.load("en_core_web_sm")

In [ ]:
API_URL = "https://api.platform.opentargets.org/api/v4/graphql"
DISEASE_ID = "MONDO_0005180"
query = f"""
query {{
  disease(efoId: "{DISEASE_ID}") {{
    associatedTargets(page: {{ index: 0, size: 100 }}) {{
      count
      rows {{
        target {{
          id
          approvedSymbol
          approvedName
        }}
        score
      }}
    }}
  }}
}}
"""
response = requests.post(API_URL, json={"query":query})
if response.status_code != 200:
    raise Exception("Something went wrong while fetching genes")
else:
    target_genes_response = response.json()
print(target_genes_response)



In [ ]:
def get_target_genes(response:dict) -> dict:
    target_res = response['data']['disease']['associatedTargets']['rows']
    genes = []
    for i in target_res:
        gene_data = {
            "gene_id": i['target']['id'],
            "symbol":  i['target']['approvedSymbol'],
            "name": i['target']['approvedName'],
            "score": i['score']
        }
        genes.append(gene_data)
    return genes

genes_target = get_target_genes(target_genes_response)
print(genes_target)

In [ ]:
genes_target_prelim_df = pd.DataFrame(genes_target)
genes_target_prelim_df

In [ ]:
def load_gtex_expression(filepath: str):
    df = pd.read_csv(filepath, sep="\t", skiprows=2, compression='gzip')
    brain_cols = [col for col in df.columns if 'Brain' in col]
    df["median_tpm_brain"] = df[brain_cols].median(axis=1)
    df = df.rename(columns={"Name": "gene_id", "Description": "symbol"})
    df["gene_id_stripped"] = df["gene_id"].str.split(".").str[0]
    return df[["gene_id_stripped", "symbol", "median_tpm_brain"]]

gene_tpm_brain_df = load_gtex_expression("GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_median_tpm (1).gct.gz")

def merge_open_targets_gtex(df_1, df_2):
    merged_df = df_1.merge(
        df_2,
        left_on="gene_id",
        right_on="gene_id_stripped",
        how="left"
    )
    merged_df = merged_df.drop(columns=["gene_id_stripped"])
    return merged_df

mereged_genes_parkinsons = merge_open_targets_gtex(genes_target_prelim_df, gene_tpm_brain_df)


In [ ]:
mereged_genes_parkinsons
def prioritize_genes(df):
    new_df = df.copy()
    new_df["median_tpm_brain"] = new_df["median_tpm_brain"].fillna(0)
    new_df["final_score"] =  np.log2(1+new_df["median_tpm_brain"]) * new_df["score"]
    new_df = new_df.sort_values(by="final_score", ascending=False)
    return new_df

final_genes_parkinsons_sorted = prioritize_genes(mereged_genes_parkinsons)


In [ ]:
final_genes_parkinsons_sorted

In [ ]:
def run_gprofiler(symbols: list):
    gp = GProfiler(return_dataframe=True)
    results = gp.profile(organism='hsapiens', query=symbols, no_evidences=False)
    return results

gprofiler_res = run_gprofiler(final_genes_parkinsons_sorted.head(50)['symbol_x'].dropna().tolist())

In [ ]:
gprofiler_res.to_csv('OUTPUT_CSV.csv', index=False)

In [ ]:
def enrich_top_200_genes(gprofiler_top):
    enriched_score = defaultdict(float)
    for _, row in gprofiler_top.iterrows():
        if (len(row["intersections"]) == 0):
            continue
        for i in row["intersections"]:
            enriched_score[i] += -np.log10(row["p_value"])/len(row["intersections"])
    return enriched_score

enriched_score = enrich_top_200_genes(gprofiler_res)


In [ ]:
enriched_score

In [ ]:
def compute_enriched_score(df_200, g_profiler_scores_dict, alpha):
    df_200_final = df_200.copy()
    df_200_final["g_profiler_scores"] = df_200_final["symbol_x"].map(g_profiler_scores_dict).fillna(0)
    df_200_final["final_gene_score"] = df_200_final["final_score"] + alpha*df_200_final["g_profiler_scores"]
    return df_200_final.sort_values(by="final_gene_score", ascending=False)

final_genes_scored = compute_enriched_score(final_genes_parkinsons_sorted.head(50), enriched_score, 1.0)

In [ ]:
final_genes_scored.head(5)

In [ ]:
# check literature hits for the genes shortlisted in Workflow 1

def check_eu_pmc(disease: str, gene: str) -> dict:
    url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
    try:
        response = requests.get(url, {
            "query": f'"{gene}" AND "{disease}"',
            "format": "json",
            "pageSize": 100
        })
        response.raise_for_status()
        data = response.json()
        hit_count = int(data.get("hitCount", 0))
        res = data.get("resultList", {}).get("result", [])
        years = [int(x["pubYear"]) for x in res if "pubYear" in x]
        fulltext_hits = sum(1 for paper in res if paper.get("hasTextMinedTerms") == "Y")
        return {
            "gene":gene,
            "hit_count":hit_count,
            "fulltext_hits":fulltext_hits,
            "mean_pub_year": np.mean(years) if years else np.nan
        }
    except Exception as e:
        raise RuntimeError(f"Error while fetching from EU PMC: {str(e)}")


In [ ]:
def check_literature_match_for_genes(gene_list: list, disease: str) -> dict:
    return [check_eu_pmc(disease, x) for x in gene_list]

genes_lit_match = check_literature_match_for_genes(final_genes_scored["symbol_x"].to_list(), "Parkinson's Disease")
genes_lit_match_df = pd.DataFrame(genes_lit_match)

In [ ]:
genes_lit_match_df.head(5)

In [ ]:
class LiteratureFetcher:

    def __init__(self, gene_list: List[str], disease_name: str, page_size: int = 20):
        self.gene_list = gene_list
        self.disease_name = disease_name
        self.page_size = page_size
        self._base_url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"

    def fetch_articles(self) -> List[Dict]:
        gene_wise_articles = []
        for gene in self.gene_list:
            query = f"{gene} AND {self.disease_name}"
            try:
                response = requests.get(self._base_url, params={
                    "query": query,
                    "format": "json",
                    "pageSize": self.page_size
                })
            except Exception as e:
                print(f"Error for gene: {gene} -> {e}")
                continue
            articles = []
            for i in response.json().get("resultList", {}).get("result", []):
                if "pmcid" in i:
                    articles.append({
                        "pmcid": i["pmcid"],
                        "pmid": i.get("pmid", ""),
                        "title": i.get("title", ""),
                        "source": i.get("source", ""),
                        "journal": i.get("journalTitle", "")
                    })
            gene_wise_articles.append({"gene":gene, "articles":articles})
        return gene_wise_articles


In [ ]:
class FullTextParser:
    """Extract all human-readable text recursively from XML, ignoring only known garbage tags."""

    def __init__(self):
        self.rejected_tags = {
            "ref-list", "ref", "table", "table-wrap-foot", "license", 
            "supplementary-material", "ack", "app", "permissions"
        }

    def parse_fulltext(self, pmcid: str) -> List[str]:
        url = f"https://www.ebi.ac.uk/europepmc/webservices/rest/{pmcid}/fullTextXML"
        response = requests.get(url, params={"format": "xml"})
        if not response.ok:
            print(f"[Warning] Could not fetch fulltext for PMCID {pmcid}")
            return []

        try:
            root = ET.fromstring(response.content)
            text_blocks = []

            for elem in root.iter():
                tag = elem.tag.lower().split('}')[-1]  # remove namespace
                if tag in self.rejected_tags:
                    continue
                raw = ''.join(elem.itertext()).strip()
                cleaned = re.sub(r'\s+', ' ', raw)
                if cleaned:
                    text_blocks.append(cleaned)

            return text_blocks

        except ET.ParseError:
            print(f"[Error] Failed to parse XML for PMCID {pmcid}")
            return []


In [ ]:
class EvidenceExtractor:
    def __init__(
        self,
        model_name: str = "ynie/roberta-large-snli_mnli_fever_anli_R1_R2_R3-nli",
        hypotheses: List[str] = None
    ):
        self._tokenizer = AutoTokenizer.from_pretrained(model_name)
        self._model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self._model.eval()
        self._device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self._model.to(self._device)
        self.nlp = spacy.load("en_core_web_sm")
        self.hypotheses = hypotheses or [
            "{gene} is associated with {disease}",
            "{gene} plays a role in {disease}",
            "{gene} is involved in the development of {disease}",
            "{gene} is a risk factor for {disease}"
        ]

    def _extract_sentences(self, text: str) -> List[str]:
        return [sent.text.strip() for sent in self.nlp(text).sents if sent.text.strip()]

    def _nli_infer(self, sentence: str, gene: str, disease: str) -> Optional[Dict[str, Any]]:
        for template in self.hypotheses:
            hypothesis = template.format(gene=gene, disease=disease)
            inputs = self._tokenizer(sentence, hypothesis, return_tensors="pt", truncation=True, padding=True).to(self._device)

            with torch.no_grad():
                outputs = self._model(**inputs)
                probs = torch.nn.functional.softmax(outputs.logits, dim=1)
                predicted = torch.argmax(probs, dim=1).item()

            label_map = {0: "entailment", 1: "neutral", 2: "contradiction"}
            if label_map[predicted] == "entailment" and probs[0][predicted] > 0.8:
                return {
                    "sentence": sentence,
                    "hypothesis": hypothesis,
                    "relation_label": label_map[predicted],
                    "score": float(probs[0][predicted])
                }
        return None

    def extract_from_text(self, text: str, gene: str, disease: str, pmcid: str = "") -> List[Dict[str, Any]]:
        evidence = []
        for sentence in self._extract_sentences(text):
            if gene.lower() in sentence.lower() and disease.lower() in sentence.lower():
                result = self._nli_infer(sentence, gene, disease)
                if result:
                    evidence.append({
                        "gene": gene,
                        "disease": disease,
                        "pmcid": pmcid,
                        **result
                    })
        return evidence


In [ ]:
class PipelineRunner:
    def __init__(self, disease_name: str, gene_list: List[str], num_workers: int = 8):
        self.disease_name = disease_name
        self.gene_list = gene_list
        self.num_workers = num_workers
        self._fetcher = LiteratureFetcher(gene_list, disease_name)
        self._parser = FullTextParser()
        self._evidence_extractor = EvidenceExtractor()

    def _process_gene_articles(self, entry: Dict[str, Any]) -> List[Dict[str, Any]]:
        gene = entry["gene"]
        all_evidence = []
        for article in entry["articles"]:
            pmcid = article["pmcid"]
            try:
                text_blocks = self._parser.parse_fulltext(pmcid)
                if not text_blocks:
                    print(f"[Warning] No fulltext found for {pmcid}")
                    continue

                full_text = " ".join(text_blocks)
                evidence = self._evidence_extractor.extract_from_text(
                    full_text, gene, self.disease_name, pmcid
                )
                if evidence:
                    print(f"✅ Evidence found in {pmcid} (sentences: {len(evidence)})")
                    all_evidence.extend(evidence)
                else:
                    print(f"[Info] No relation found in {pmcid}")
            except Exception as e:
                print(f"[Error] Failed to process {pmcid}: {e}")
            time.sleep(0.3)  # Respectful API pacing
        return all_evidence

    def run(self) -> List[Dict[str, Any]]:
        all_gene_articles = self._fetcher.fetch_articles()

        with ThreadPool(self.num_workers) as pool:
            results = list(tqdm(
                pool.imap(self._process_gene_articles, all_gene_articles),
                total=len(all_gene_articles),
                desc="🔍 Processing Genes in Literature"
            ))

        return [item for sublist in results for item in sublist]

In [ ]:
runner = PipelineRunner("Parkinson's disease", final_genes_scored.head(25)["symbol_x"].to_list(), num_workers=9)
results = runner.run()

In [ ]:
results